# GVH Diagonal Cubic 0.2.23.2 — Covariant Source Projector Derivation

**Auteur : Charlemagne O Laurince**

---

## Objectif

Le notebook `0.2.23.1` a identifié le premier verrou de la branche weak-field non triviale :

\[
\boxed{\Pi_{\mu\nu}[T]}
\]

Le présent notebook cherche une définition covariante précise du projecteur de source compatible avec le secteur directionnel spatial de GVH.

La construction doit satisfaire :

\[
\Pi_{\mu\nu}=\Pi_{\nu\mu},
\]

\[
u^\mu\Pi_{\mu\nu}=0,
\]

\[
g^{\mu\nu}\Pi_{\mu\nu}=0,
\]

et isoler la partie anisotrope du tenseur énergie-impulsion.

---

## Hypothèse structurelle reprise des notebooks précédents

Les notebooks `0.2.12–0.2.23.1` traitent le secteur directionnel comme un secteur principalement spatial et sans trace.

Pour le rendre covariant, il faut introduire un champ temporel unitaire :

\[
u^\mu u_\mu=-1.
\]

Le projecteur spatial associé est :

\[
h_{\mu\nu}=g_{\mu\nu}+u_\mu u_\nu.
\]

La partie symétrique spatiale sans trace d’un tenseur \(T_{\alpha\beta}\) est alors obtenue avec le projecteur de rang quatre :

\[
\boxed{
\mathcal P_{\mu\nu}{}^{\alpha\beta}
=
h_{(\mu}{}^\alpha h_{\nu)}{}^\beta
-
\frac13 h_{\mu\nu}h^{\alpha\beta}
}
\]

et :

\[
\boxed{
\Pi_{\mu\nu}[T]
=
\mathcal P_{\mu\nu}{}^{\alpha\beta}
T_{\alpha\beta}
}.
\]

---

## Limite scientifique

Cette construction est unique une fois les hypothèses suivantes fixées :

1. espace-temps à quatre dimensions ;
2. signature \((-+++)\) ;
3. champ temporel unitaire \(u^\mu\) ;
4. secteur source spatial, symétrique et sans trace ;
5. projection locale et linéaire en \(T_{\mu\nu}\).

Elle n’est pas une conséquence automatique de la covariance seule.  
Le choix ou la dynamique de \(u^\mu\) doit encore être relié à l’action GVH complète.

---

## Statuts possibles

```text
PASS-SPATIAL-STF-SOURCE-PROJECTOR
PASS-ANISOTROPIC-SOURCE-CHANNEL
BLOCKED-PREFERRED-TIME-FIELD-DYNAMICS
BLOCKED-FULL-ACTION-CONSISTENCY
```

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sympy as sp

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 200)

print("Python :", sys.version)
print("SymPy :", sp.__version__)

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
SymPy : 1.14.0


# 1. Dépôt et chemins

In [2]:
REPOSITORY_URL = "https://github.com/col38470682/Univers.git"
REPOSITORY_DIR = Path("/content/Univers")
PROJECT_ROOT = REPOSITORY_DIR / "gvh_diagonal_cubic"

if (REPOSITORY_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(PROJECT_ROOT)

PROCESSED_SOURCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "source_projector"
)

EXPORT_DIR = PROJECT_ROOT / "exports"

for directory in [
    PROCESSED_SOURCE_DIR,
    EXPORT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("PROJECT_ROOT :", PROJECT_ROOT)

PROJECT_ROOT : /content/Univers/gvh_diagonal_cubic


# 2. Dépendances théoriques obligatoires

In [3]:
NOTEBOOK_PATTERNS = {
    "0.2.21": "*0.2.21*Equivalence*Principle*PPN*Constraints*.ipynb",
    "0.2.22": "*0.2.22*Static*Spherical*PPN*Derivation*.ipynb",
    "0.2.23": "*0.2.23*Covariant*Static*Field*Source*Matching*.ipynb",
    "0.2.23.1": "*0.2.23.1*Weak*Field*Coefficient*Extraction*PPN*.ipynb",
}

resolved_notebooks = {}

for notebook_id, pattern in NOTEBOOK_PATTERNS.items():
    matches = sorted(
        PROJECT_ROOT.rglob(pattern)
    )

    resolved_notebooks[
        notebook_id
    ] = (
        matches[0]
        if matches
        else None
    )

source_notebooks_df = pd.DataFrame([
    {
        "notebook_id": notebook_id,
        "resolved_path": (
            str(path)
            if path is not None
            else ""
        ),
        "found": path is not None,
    }
    for notebook_id, path
    in resolved_notebooks.items()
])

source_notebooks_df

,notebook_id,resolved_path,found
0,0.2.21,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2A_theoretical_foundations/0.2A_theoretical_foundations/GVH_Diagonal_Cubic_0.2.21_Equivalence_Principle_PPN_Constraints.ipynb,True
1,0.2.22,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2A_theoretical_foundations/0.2A_theoretical_foundations/GVH_Diagonal_Cubic_0.2.22_Static_Spherical_Solution_PPN_Derivation.ipynb,True
2,0.2.23,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2A_theoretical_foundations/0.2A_theoretical_foundations/GVH_Diagonal_Cubic_0.2.23_Covariant_Static_Field_Equations_Source_Matching.ipynb,True
3,0.2.23.1,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2A_theoretical_foundations/0.2A_theoretical_foundations/GVH_Diagonal_Cubic_0.2.23.1_Weak_Field_Coefficient_Extraction_for_PPN_UPDATED.ipynb,True


In [4]:
all_sources_found = bool(
    source_notebooks_df[
        "found"
    ].all()
)

if not all_sources_found:
    print(
        "ATTENTION : certaines dépendances ne sont pas trouvées. "
        "La dérivation algébrique reste exécutable, mais la chaîne "
        "d'archivage n'est pas complète."
    )

# 3. Décomposition covariante de \(T_{\mu
u}\)

Relativement à \(u^\mu\), un tenseur énergie-impulsion symétrique se décompose sous la forme :

\[
T_{\mu\nu}
=
\rho u_\mu u_\nu
+
p h_{\mu\nu}
+
2u_{(\mu}q_{\nu)}
+
\pi_{\mu\nu},
\]

avec :

\[
u^\mu q_\mu=0,
\]

\[
u^\mu\pi_{\mu\nu}=0,
\]

\[
\pi^\mu{}_\mu=0.
\]

La quantité recherchée est précisément :

\[
\boxed{
\Pi_{\mu\nu}[T]=\pi_{\mu\nu}
}.
\]

Le projecteur ne couple donc pas directement à la densité isotrope \(\rho\) ni à la pression isotrope \(p\). Il couple au stress anisotrope.

In [5]:
projector_definition_df = pd.DataFrame([
    {
        "object": "spatial projector",
        "symbol": "h_mn",
        "definition": "g_mn + u_m u_n",
        "role": "orthogonal projection to u^m",
    },
    {
        "object": "rank-4 STF projector",
        "symbol": "P_mn^ab",
        "definition": (
            "h_(m^a h_n)^b - (1/3) h_mn h^ab"
        ),
        "role": (
            "symmetric spatial trace-free projection"
        ),
    },
    {
        "object": "GVH source channel",
        "symbol": "Pi_mn[T]",
        "definition": "P_mn^ab T_ab",
        "role": "anisotropic stress source",
    },
])

projector_definition_df

,object,symbol,definition,role
0,spatial projector,h_mn,g_mn + u_m u_n,orthogonal projection to u^m
1,rank-4 STF projector,P_mn^ab,h_(m^a h_n)^b - (1/3) h_mn h^ab,symmetric spatial trace-free projection
2,GVH source channel,Pi_mn[T],P_mn^ab T_ab,anisotropic stress source


# 4. Vérification explicite en repère local de Minkowski

In [6]:
rho, p = sp.symbols(
    "rho p",
    real=True,
)

p_r, p_t = sp.symbols(
    "p_r p_t",
    real=True,
)

eta = sp.diag(
    -1,
    1,
    1,
    1,
)

u_up = sp.Matrix([
    1,
    0,
    0,
    0,
])

u_down = eta * u_up

h_down = (
    eta
    + u_down * u_down.T
)

h_up = (
    eta.inv()
    + u_up * u_up.T
)

h_mixed = h_down * eta.inv()

print("u^mu u_mu =", (u_up.T * u_down)[0])
print("h_mn =")
sp.Matrix(h_down)

u^mu u_mu = -1
h_mn =


Matrix([
[0, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]])

# 5. Fonction de projection STF spatiale

In [7]:
def spatial_stf_projection(T_down):
    T_down = sp.Matrix(T_down)

    spatial_projection = (
        h_mixed
        * T_down
        * h_mixed.T
    )

    spatial_trace = sp.simplify(
        sum(
            h_up[i, j]
            * T_down[i, j]
            for i in range(4)
            for j in range(4)
        )
    )

    Pi_down = sp.simplify(
        spatial_projection
        - sp.Rational(1, 3)
        * h_down
        * spatial_trace
    )

    return Pi_down


def tensor_trace(T_down):
    T_down = sp.Matrix(T_down)

    return sp.simplify(
        sum(
            eta.inv()[i, j]
            * T_down[i, j]
            for i in range(4)
            for j in range(4)
        )
    )


def temporal_contraction(T_down):
    T_down = sp.Matrix(T_down)

    return sp.simplify(
        u_up.T
        * T_down
    )

# 6. Fluide parfait isotrope

In [8]:
T_perfect = sp.diag(
    rho,
    p,
    p,
    p,
)

Pi_perfect = spatial_stf_projection(
    T_perfect
)

perfect_fluid_df = pd.DataFrame([{
    "source": "perfect isotropic fluid",
    "T_mn": str(T_perfect),
    "Pi_mn": str(Pi_perfect),
    "Pi_zero": bool(
        Pi_perfect == sp.zeros(4)
    ),
}])

perfect_fluid_df

,source,T_mn,Pi_mn,Pi_zero
0,perfect isotropic fluid,"Matrix([[rho, 0, 0, 0], [0, p, 0, 0], [0, 0, p, 0], [0, 0, 0, p]])","Matrix([[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]])",True


In [9]:
perfect_symmetry_pass = bool(
    Pi_perfect
    == Pi_perfect.T
)

perfect_trace_pass = bool(
    tensor_trace(
        Pi_perfect
    ) == 0
)

perfect_orthogonality_pass = bool(
    temporal_contraction(
        Pi_perfect
    ) == sp.zeros(1, 4)
)

perfect_projector_tests_df = pd.DataFrame([
    {
        "test": "symmetric",
        "pass": perfect_symmetry_pass,
    },
    {
        "test": "trace free",
        "pass": perfect_trace_pass,
    },
    {
        "test": "u-orthogonal",
        "pass": perfect_orthogonality_pass,
    },
    {
        "test": "perfect fluid projects to zero",
        "pass": bool(
            Pi_perfect == sp.zeros(4)
        ),
    },
])

perfect_projector_tests_df

,test,pass
0,symmetric,True
1,trace free,True
2,u-orthogonal,True
3,perfect fluid projects to zero,True


# 7. Fluide sphérique anisotrope

In [10]:
T_anisotropic = sp.diag(
    rho,
    p_r,
    p_t,
    p_t,
)

Pi_anisotropic = sp.simplify(
    spatial_stf_projection(
        T_anisotropic
    )
)

Pi_anisotropic

Matrix([
[0,                 0,              0,              0],
[0, 2*p_r/3 - 2*p_t/3,              0,              0],
[0,                 0, -p_r/3 + p_t/3,              0],
[0,                 0,              0, -p_r/3 + p_t/3]])

In [11]:
Delta_p = sp.symbols(
    "Delta_p",
    real=True,
)

Pi_anisotropic_delta = sp.simplify(
    Pi_anisotropic.subs(
        p_r,
        p_t + Delta_p,
    )
)

expected_anisotropic = sp.diag(
    0,
    sp.Rational(2, 3) * Delta_p,
    -sp.Rational(1, 3) * Delta_p,
    -sp.Rational(1, 3) * Delta_p,
)

anisotropic_formula_pass = bool(
    sp.simplify(
        Pi_anisotropic_delta
        - expected_anisotropic
    )
    == sp.zeros(4)
)

anisotropic_result_df = pd.DataFrame([{
    "anisotropy": "Delta_p = p_r - p_t",
    "Pi_tt": "0",
    "Pi_rr": "2 Delta_p / 3",
    "Pi_theta_theta_local": "-Delta_p / 3",
    "Pi_phi_phi_local": "-Delta_p / 3",
    "formula_pass": anisotropic_formula_pass,
}])

anisotropic_result_df

,anisotropy,Pi_tt,Pi_rr,Pi_theta_theta_local,Pi_phi_phi_local,formula_pass
0,Delta_p = p_r - p_t,0,2 Delta_p / 3,-Delta_p / 3,-Delta_p / 3,True


Le résultat est :

\[
\boxed{
\Pi_{\hat\mu\hat\nu}
=
\operatorname{diag}
\left(
0,
\frac23\Delta p,
-\frac13\Delta p,
-\frac13\Delta p
\right)
}
\]

où :

\[
\Delta p=p_r-p_t.
\]

Ainsi :

\[
p_r=p_t
\quad\Longrightarrow\quad
\Pi_{\mu\nu}=0,
\]

mais :

\[
p_r\neq p_t
\quad\Longrightarrow\quad
\Pi_{\mu\nu}\neq0.
\]

# 8. Idempotence du projecteur

In [12]:
X00, X01, X02, X03 = sp.symbols(
    "X00 X01 X02 X03",
    real=True,
)

X11, X12, X13 = sp.symbols(
    "X11 X12 X13",
    real=True,
)

X22, X23, X33 = sp.symbols(
    "X22 X23 X33",
    real=True,
)

X = sp.Matrix([
    [X00, X01, X02, X03],
    [X01, X11, X12, X13],
    [X02, X12, X22, X23],
    [X03, X13, X23, X33],
])

Pi_X = spatial_stf_projection(
    X
)

Pi_Pi_X = spatial_stf_projection(
    Pi_X
)

idempotence_pass = bool(
    sp.simplify(
        Pi_Pi_X - Pi_X
    )
    == sp.zeros(4)
)

generic_projector_tests_df = pd.DataFrame([
    {
        "test": "idempotence P(P(X)) = P(X)",
        "pass": idempotence_pass,
    },
    {
        "test": "generic projected tensor symmetric",
        "pass": bool(
            Pi_X == Pi_X.T
        ),
    },
    {
        "test": "generic projected tensor trace free",
        "pass": bool(
            tensor_trace(
                Pi_X
            ) == 0
        ),
    },
    {
        "test": "generic projected tensor u-orthogonal",
        "pass": bool(
            temporal_contraction(
                Pi_X
            )
            == sp.zeros(1, 4)
        ),
    },
])

generic_projector_tests_df

,test,pass
0,idempotence P(P(X)) = P(X),True
1,generic projected tensor symmetric,True
2,generic projected tensor trace free,True
3,generic projected tensor u-orthogonal,True


# 9. Nombre de composantes projetées

In [13]:
independent_spatial_symmetric = 6
trace_constraint_count = 1
stf_component_count = (
    independent_spatial_symmetric
    - trace_constraint_count
)

component_count_df = pd.DataFrame([{
    "spatial_symmetric_components": (
        independent_spatial_symmetric
    ),
    "trace_constraints": (
        trace_constraint_count
    ),
    "spatial_STF_components": (
        stf_component_count
    ),
    "expected_spin2_like_components": 5,
    "count_pass": (
        stf_component_count == 5
    ),
}])

component_count_df

,spatial_symmetric_components,trace_constraints,spatial_STF_components,expected_spin2_like_components,count_pass
0,6,1,5,5,True


# 10. Unicité sous les hypothèses fixées

Considérons le projecteur local linéaire le plus simple construit uniquement avec \(h_{\mu\nu}\) :

\[
\mathcal P_{\mu\nu}{}^{\alpha\beta}
=
A\,h_{(\mu}{}^\alpha h_{\nu)}{}^\beta
+
B\,h_{\mu\nu}h^{\alpha\beta}.
\]

La conservation de la partie spatiale symétrique impose :

\[
A=1.
\]

La condition sans trace impose :

\[
h^{\mu\nu}
\mathcal P_{\mu\nu}{}^{\alpha\beta}
=0.
\]

En trois dimensions spatiales :

\[
1+3B=0,
\]

d’où :

\[
\boxed{B=-\frac13}.
\]

In [14]:
A, B = sp.symbols(
    "A B",
    real=True,
)

uniqueness_solution = sp.solve(
    [
        sp.Eq(A, 1),
        sp.Eq(A + 3 * B, 0),
    ],
    [
        A,
        B,
    ],
    dict=True,
)

uniqueness_df = pd.DataFrame([{
    "ansatz": (
        "A h_(m^a h_n)^b + B h_mn h^ab"
    ),
    "solution": str(
        uniqueness_solution
    ),
    "A": "1",
    "B": "-1/3",
    "scope": (
        "unique local linear spatial STF projector "
        "for fixed u^mu in 3 spatial dimensions"
    ),
}])

uniqueness_df

,ansatz,solution,A,B,scope
0,A h_(m^a h_n)^b + B h_mn h^ab,"[{A: 1, B: -1/3}]",1,-1/3,unique local linear spatial STF projector for fixed u^mu in 3 spatial dimensions


# 11. Comparaison avec le projecteur 4D sans trace

Un autre objet covariant possible est la partie quatre-dimensionnelle sans trace :

\[
T_{\mu\nu}
-
\frac14g_{\mu\nu}T.
\]

Mais ce projecteur ne sélectionne pas uniquement le secteur spatial directionnel.  
Pour un fluide parfait isotrope, il ne s’annule généralement pas.

Il représente donc une autre théorie de couplage, pas une simple réécriture du projecteur spatial STF.

In [15]:
T_trace_4d = tensor_trace(
    T_perfect
)

Pi_4d_perfect = sp.simplify(
    T_perfect
    - sp.Rational(1, 4)
    * eta
    * T_trace_4d
)

projector_comparison_df = pd.DataFrame([
    {
        "projector": "spatial STF relative to u^mu",
        "perfect_fluid_zero": bool(
            Pi_perfect == sp.zeros(4)
        ),
        "requires_u_field": True,
        "selects_anisotropic_stress": True,
        "compatible_with_current_GVH_spatial_sector": True,
    },
    {
        "projector": "4D trace-free",
        "perfect_fluid_zero": bool(
            Pi_4d_perfect == sp.zeros(4)
        ),
        "requires_u_field": False,
        "selects_anisotropic_stress": False,
        "compatible_with_current_GVH_spatial_sector": False,
    },
])

projector_comparison_df

,projector,perfect_fluid_zero,requires_u_field,selects_anisotropic_stress,compatible_with_current_GVH_spatial_sector
0,spatial STF relative to u^mu,True,True,True,True
1,4D trace-free,False,False,False,False


# 12. Conservation et cohérence dynamique

Même si :

\[
\nabla^\mu T_{\mu\nu}=0,
\]

il ne s’ensuit pas automatiquement que :

\[
\nabla^\mu\Pi_{\mu\nu}[T]=0.
\]

En effet, \(\Pi_{\mu\nu}\) dépend aussi de \(u^\mu\) et de ses dérivées.

Le couplage :

\[
S_{\rm int}
=
\lambda_T
\int d^4x\sqrt{-g}\,
D_{\mu\nu}\Pi^{\mu\nu}[T]
\]

peut être écrit covariamment, mais sa cohérence complète exige de préciser :

1. si \(u^\mu\) est un champ dynamique, une variable de matière ou un champ dérivé ;
2. la contrainte \(u^\mu u_\mu=-1\) ;
3. la variation de \(\Pi_{\mu\nu}\) par rapport à \(g_{\mu\nu}\) ;
4. la variation par rapport à \(u^\mu\) ;
5. le bilan d’énergie-impulsion du système matière + \(D_{\mu\nu}\) + \(u^\mu\).

Le projecteur algébrique est donc dérivé, mais la fermeture dynamique complète reste une étape distincte.

In [16]:
dynamic_closure_df = pd.DataFrame([
    {
        "requirement": "definition of u^mu",
        "status": "NOT_FIXED GLOBALLY",
        "needed_for": "physical covariance and frame interpretation",
    },
    {
        "requirement": "normalization constraint u^2=-1",
        "status": "FORMALLY REQUIRED",
        "needed_for": "projector consistency",
    },
    {
        "requirement": "metric variation of Pi_mn[T]",
        "status": "NOT DERIVED",
        "needed_for": "Einstein-equation backreaction",
    },
    {
        "requirement": "u-field variation",
        "status": "NOT DERIVED",
        "needed_for": "preferred-frame dynamics",
    },
    {
        "requirement": "total stress-energy conservation",
        "status": "NOT DERIVED",
        "needed_for": "full action consistency",
    },
])

dynamic_closure_df

,requirement,status,needed_for
0,definition of u^mu,NOT_FIXED GLOBALLY,physical covariance and frame interpretation
1,normalization constraint u^2=-1,FORMALLY REQUIRED,projector consistency
2,metric variation of Pi_mn[T],NOT DERIVED,Einstein-equation backreaction
3,u-field variation,NOT DERIVED,preferred-frame dynamics
4,total stress-energy conservation,NOT DERIVED,full action consistency


# 13. Artefact du projecteur candidat

In [17]:
projector_artifact = {
    "artifact_status": (
        "VALIDATED_ALGEBRAIC_PROJECTOR_"
        "DYNAMIC_CLOSURE_PENDING"
    ),
    "projector_id": "spatial_symmetric_trace_free",
    "spacetime_dimension": 4,
    "spatial_dimension": 3,
    "metric_signature": "-+++",
    "required_structure": {
        "timelike_field": "u^mu",
        "normalization": "u^mu u_mu = -1",
        "spatial_metric": "h_mn = g_mn + u_m u_n",
    },
    "rank4_projector": (
        "P_mn^ab = h_(m^a h_n)^b "
        "- (1/3) h_mn h^ab"
    ),
    "source_projection": (
        "Pi_mn[T] = P_mn^ab T_ab"
    ),
    "properties": {
        "symmetric": True,
        "spatial": True,
        "trace_free": True,
        "idempotent": True,
        "perfect_isotropic_fluid_zero": True,
        "anisotropic_fluid_nonzero": True,
    },
    "anisotropic_spherical_source": {
        "Delta_p": "p_r - p_t",
        "orthonormal_components": (
            "diag(0, 2 Delta_p/3, "
            "-Delta_p/3, -Delta_p/3)"
        ),
    },
    "scope": (
        "Unique local linear spatial STF projector "
        "once u^mu is fixed."
    ),
    "not_yet_derived": [
        "dynamics or physical origin of u^mu",
        "metric variation of the interaction",
        "u-field equation",
        "total conservation law",
        "metric backreaction coefficients alpha_t and alpha_s",
    ],
    "source_notebooks": [
        "0.2.21",
        "0.2.22",
        "0.2.23",
        "0.2.23.1",
    ],
}

projector_artifact

{'artifact_status': 'VALIDATED_ALGEBRAIC_PROJECTOR_DYNAMIC_CLOSURE_PENDING',
 'projector_id': 'spatial_symmetric_trace_free',
 'spacetime_dimension': 4,
 'spatial_dimension': 3,
 'metric_signature': '-+++',
 'required_structure': {'timelike_field': 'u^mu',
  'normalization': 'u^mu u_mu = -1',
  'spatial_metric': 'h_mn = g_mn + u_m u_n'},
 'rank4_projector': 'P_mn^ab = h_(m^a h_n)^b - (1/3) h_mn h^ab',
 'source_projection': 'Pi_mn[T] = P_mn^ab T_ab',
 'properties': {'symmetric': True,
  'spatial': True,
  'trace_free': True,
  'idempotent': True,
  'perfect_isotropic_fluid_zero': True,
  'anisotropic_fluid_nonzero': True},
 'anisotropic_spherical_source': {'Delta_p': 'p_r - p_t',
  'orthonormal_components': 'diag(0, 2 Delta_p/3, -Delta_p/3, -Delta_p/3)'},
 'scope': 'Unique local linear spatial STF projector once u^mu is fixed.',
 'not_yet_derived': ['dynamics or physical origin of u^mu',
  'metric variation of the interaction',
  'u-field equation',
  'total conservation law',
  'metric

# 14. Portes de validation

In [18]:
algebraic_projector_pass = bool(
    perfect_projector_tests_df[
        "pass"
    ].all()
    and generic_projector_tests_df[
        "pass"
    ].all()
    and anisotropic_formula_pass
    and component_count_df.loc[
        0,
        "count_pass",
    ]
)

preferred_time_field_dynamics_derived = False
metric_variation_derived = False
total_conservation_derived = False

full_action_consistency_pass = bool(
    preferred_time_field_dynamics_derived
    and metric_variation_derived
    and total_conservation_derived
)

validation_gates_df = pd.DataFrame([
    {
        "gate": "source notebooks archived",
        "pass": all_sources_found,
    },
    {
        "gate": "algebraic STF projector",
        "pass": algebraic_projector_pass,
    },
    {
        "gate": "perfect-fluid null channel",
        "pass": bool(
            Pi_perfect == sp.zeros(4)
        ),
    },
    {
        "gate": "anisotropic source channel",
        "pass": anisotropic_formula_pass,
    },
    {
        "gate": "preferred-time-field dynamics",
        "pass": preferred_time_field_dynamics_derived,
    },
    {
        "gate": "metric variation",
        "pass": metric_variation_derived,
    },
    {
        "gate": "total action consistency",
        "pass": full_action_consistency_pass,
    },
])

validation_gates_df

,gate,pass
0,source notebooks archived,True
1,algebraic STF projector,True
2,perfect-fluid null channel,True
3,anisotropic source channel,True
4,preferred-time-field dynamics,False
5,metric variation,False
6,total action consistency,False


# 15. Décision scientifique

In [19]:
if not algebraic_projector_pass:
    FINAL_STATUS = (
        "BLOCKED-SOURCE-PROJECTOR-ALGEBRA"
    )
elif full_action_consistency_pass:
    FINAL_STATUS = (
        "PASS-COVARIANT-SOURCE-PROJECTOR-"
        "AND-FULL-ACTION-CONSISTENCY"
    )
else:
    FINAL_STATUS = (
        "PASS-SPATIAL-STF-SOURCE-PROJECTOR_"
        "PASS-ANISOTROPIC-SOURCE-CHANNEL_"
        "BLOCKED-PREFERRED-TIME-FIELD-DYNAMICS_"
        "BLOCKED-FULL-ACTION-CONSISTENCY"
    )

decision_df = pd.DataFrame([{
    "final_status": FINAL_STATUS,
    "algebraic_projector_pass": algebraic_projector_pass,
    "perfect_fluid_channel_zero": bool(
        Pi_perfect == sp.zeros(4)
    ),
    "anisotropic_channel_nonzero": anisotropic_formula_pass,
    "full_action_consistency_pass": (
        full_action_consistency_pass
    ),
    "next_required_derivation": (
        "derive the dynamical status of u^mu and "
        "vary S_int with respect to g_mn and u^mu"
    ),
}])

print("STATUT FINAL :", FINAL_STATUS)
decision_df

STATUT FINAL : PASS-SPATIAL-STF-SOURCE-PROJECTOR_PASS-ANISOTROPIC-SOURCE-CHANNEL_BLOCKED-PREFERRED-TIME-FIELD-DYNAMICS_BLOCKED-FULL-ACTION-CONSISTENCY


,final_status,algebraic_projector_pass,perfect_fluid_channel_zero,anisotropic_channel_nonzero,full_action_consistency_pass,next_required_derivation
0,PASS-SPATIAL-STF-SOURCE-PROJECTOR_PASS-ANISOTROPIC-SOURCE-CHANNEL_BLOCKED-PREFERRED-TIME-FIELD-DYNAMICS_BLOCKED-FULL-ACTION-CONSISTENCY,True,True,True,False,derive the dynamical status of u^mu and vary S_int with respect to g_mn and u^mu


# 16. Exports

In [20]:
PREFIX = "GVH_Diagonal_Cubic_0.2.23.2"

exports = {
    "Source_Notebooks": source_notebooks_df,
    "Projector_Definition": projector_definition_df,
    "Perfect_Fluid": perfect_fluid_df,
    "Perfect_Projector_Tests": perfect_projector_tests_df,
    "Anisotropic_Result": anisotropic_result_df,
    "Generic_Projector_Tests": generic_projector_tests_df,
    "Component_Count": component_count_df,
    "Uniqueness": uniqueness_df,
    "Projector_Comparison": projector_comparison_df,
    "Dynamic_Closure": dynamic_closure_df,
    "Validation_Gates": validation_gates_df,
    "Decision": decision_df,
}

for suffix, table in exports.items():
    table.to_csv(
        EXPORT_DIR
        / f"{PREFIX}_{suffix}.csv",
        index=False,
    )

PROJECTOR_FILE = (
    PROCESSED_SOURCE_DIR
    / "gvh_covariant_source_projector.json"
)

PROJECTOR_FILE.write_text(
    json.dumps(
        projector_artifact,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

metadata = {
    "notebook": (
        "GVH_Diagonal_Cubic_0.2.23.2_"
        "Covariant_Source_Projector_Derivation"
    ),
    "execution_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "final_status": FINAL_STATUS,
    "algebraic_projector_pass": (
        algebraic_projector_pass
    ),
    "projector_file": str(
        PROJECTOR_FILE
    ),
    "preferred_time_field_dynamics_derived": (
        preferred_time_field_dynamics_derived
    ),
    "metric_variation_derived": (
        metric_variation_derived
    ),
    "full_action_consistency_pass": (
        full_action_consistency_pass
    ),
    "next_action": (
        "Construct 0.2.23.3 to define or derive u^mu, "
        "its normalization mechanism, and the variation "
        "of the interaction action."
    ),
}

with (
    EXPORT_DIR
    / f"{PREFIX}_Metadata.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Projector artifact :", PROJECTOR_FILE)
print("Exports :", EXPORT_DIR)

Projector artifact : /content/Univers/gvh_diagonal_cubic/data/processed/source_projector/gvh_covariant_source_projector.json
Exports : /content/Univers/gvh_diagonal_cubic/exports


# Conclusion

Le projecteur algébrique covariant compatible avec le secteur spatial sans trace est :

\[
\boxed{
\Pi_{\mu\nu}[T]
=
\left(
h_{(\mu}{}^\alpha h_{\nu)}{}^\beta
-
\frac13h_{\mu\nu}h^{\alpha\beta}
\right)
T_{\alpha\beta}
}
\]

avec :

\[
h_{\mu\nu}=g_{\mu\nu}+u_\mu u_\nu.
\]

Il vérifie :

\[
\Pi_{\mu\nu}=\Pi_{\nu\mu},
\]

\[
u^\mu\Pi_{\mu\nu}=0,
\]

\[
\Pi^\mu{}_\mu=0,
\]

\[
\mathcal P^2=\mathcal P.
\]

Pour un fluide parfait isotrope :

\[
\boxed{\Pi_{\mu\nu}=0}.
\]

Pour un fluide sphérique anisotrope :

\[
\boxed{
\Pi_{\hat\mu\hat\nu}
=
\operatorname{diag}
\left(
0,
\frac23(p_r-p_t),
-\frac13(p_r-p_t),
-\frac13(p_r-p_t)
\right)
}.
\]

Ainsi, le premier verrou de `0.2.23.1` est levé au niveau algébrique.

Le verrou suivant concerne la nature de \(u^\mu\), la variation complète du couplage et la conservation totale. Ces éléments sont nécessaires avant de dériver rigoureusement :

\[
\alpha_t,
\qquad
\alpha_s.
\]